Run this code in a code cell
 before answering any of the question. This will create your knowledge
base and the FAISS index for this milestone.

In [1]:
!pip install faiss-cpu #install FAISS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 79.4 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv('/content/train.csv')

In [3]:
print("Creating Knwoledge base...")
kb = []
for index, row in train.iterrows():
  correct_letter = row["answer"]
  kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index...")
model = SentenceTransformer('all-MiniLM-L6-v2')

kb_embeddings = model.encode(kb, show_progress_bar = False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created....")

Creating Knwoledge base...
Loading embedding model and creating index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created....


Zero-shot classifier for Q1, Q2, Q6:

In [4]:
from transformers import pipeline
zs = pipeline("zero-shot-classification", model = "facebook/bart-large-mnli")
row_150 = train.iloc[150]
prompt_150 = str(row_150["prompt"])
labels_150 = [str(row_150["A"]), str(row_150["B"]), str(row_150["C"]), str(row_150["D"]), str(row_150["E"])]
ans_150    = str(row_150[row_150["answer"]])

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Q1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)

In [5]:
prediction = zs(prompt_150, candidate_labels = labels_150)

In [6]:
for idx, label in enumerate(prediction["labels"]):
  if label == ans_150:
    print(f"Probability score assigned to ground-truth correct option: {prediction["scores"][idx]:.3f}")

Probability score assigned to ground-truth correct option: 0.384


Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [22]:
prompt_150_embedding = model.encode(prompt_150)
prompt_150_embedding = prompt_150_embedding.reshape(1,-1)
distances, indices = index.search(prompt_150_embedding, k=10)
print(distances)

[[0.26394257 0.26394257 0.2664284  0.2664284  0.2686218  0.26993614
  0.26993614 0.26993614 0.28316098 0.28714597]]


In [8]:
for i in range(len(indices[0])):
  if indices[0][i] == 150:
    print("Rank of the true correct document:", i+1)

Rank of the true correct document: 10


Q3. Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [9]:
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in indices[0]]

pairs = [[prompt_150,doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [10]:
print(ce_scores)
print(indices[0])

[4.6602316 4.6602316 4.737523  4.737523  4.687043  4.752577  4.752577
 4.752577  4.7072477 4.758512 ]
[ 663 1701 1269 1532  576  847 1693 1906  168  150]


In [11]:
scores = []
for i in range(len(indices[0])):
  scores.append((ce_scores[i], indices[0][i]))
scores.sort(reverse=True)
for i in range(len(scores)):
  if scores[i][1] == 150:
    print("Rank of the true correct document:" , i+1)


Rank of the true correct document: 1


Q4. Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?

In [12]:
row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])
labels_42 = [str(row_42["A"]), str(row_42["B"]), str(row_42["C"]), str(row_42["D"]), str(row_42["E"])]
answer_42 = str(row_42[row_42["answer"]])

In [23]:
prompt_42_embedding = model.encode(prompt_42).reshape(1,-1)
distances_5, indices_5 = index.search(prompt_42_embedding, k=5)
concatenated_docs = []
for doc in indices_5[0]:
  concatenated_docs.append(kb[doc])

concatenated_docs = " ".join(concatenated_docs)

system_prompt = f"Context: {concatenated_docs} Question: {prompt_42}"

In [14]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
input_tokens = tokenizer(system_prompt, truncation=False, return_tensors = "pt")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [15]:
print(f"Number of tokens generated: {input_tokens["input_ids"].shape[1]}")

Number of tokens generated: 216


Q5. Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).

In [16]:
row_index = 150
true_doc = kb[row_index]
rag_string = f"Context: {true_doc} Question: {prompt_150}"

prediction = zs(rag_string, candidate_labels = labels_150)
prediction

{'sequence': 'Context: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos." Question: Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.',
 'labels': ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."',
  'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."',
  'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effec

In [17]:
for idx, label in enumerate(prediction["labels"]):
  if label == ans_150:
    print(f"Probability score assigned to ground-truth correct option: {prediction["scores"][idx]:.3f}")

Probability score assigned to ground-truth correct option: 0.989


Q6. What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).

In [73]:
false_doc = kb[999]
rag_string = f"Context: {false_doc} Question: {prompt_150}"

prediction = zs(rag_string, candidate_labels = labels_150)
for idx,label in enumerate(prediction["labels"]):
  if label == ans_150:
    print(f"Probability score assigned to ground-truth correct option: {prediction['scores'][idx]:.3f}")

Probability score assigned to ground-truth correct option: 0.529


Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).

In [72]:
hit = 0
for i in range(100):
  row = train.iloc[i]
  prompt = str(row["prompt"])
  answer_letter = str(row["answer"])
  correct_text  = str(row[answer_letter])

  prompt_embedding = model.encode(prompt).reshape(1,-1)
  distances, indices = index.search(prompt_embedding, k=5)

  docs_5 = [kb[idx] for idx in indices[0]]

  for doc in docs_5:
    if correct_text in doc:
      hit +=1
      break

print(f"Hit Rate Percentage: {hit} %")


Hit Rate Percentage: 73 %


Q8. Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
For each row, your pipeline must do the following in order:

Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

Score: Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).

In [63]:
def calculate_map(options, truth):
  options = options.split()

  if truth == options[0]:
    return 1.000
  elif truth == options[1]:
    return 0.500
  elif truth == options[2]:
    return 0.333
  else:
    return 0.000

In [70]:
from tqdm.notebook import tqdm
score = 0

for i in tqdm(range(20)):
  row = train.iloc[i]
  prompt = str(row["prompt"])
  answer = str(row["answer"])
  labels = [str(row["A"]), str(row["B"]), str(row["C"]), str(row["D"]), str(row["E"])]
  prompt_embedding = model.encode(prompt).reshape(1,-1)

  distances, indices = index.search(prompt_embedding, k=5)

  docs_5 = [kb[j] for j in indices[0]]
  pairs = [[prompt,doc] for doc in docs_5]

  ce_score = cross_encoder.predict(pairs)
  best_doc = docs_5[ce_score.argmax()]

  rag_string = f"Context: {best_doc} Question: {prompt}"

  prediction = zs(rag_string, candidate_labels = labels)

  index_to_option = {0: "A", 1: "B", 2:"C", 3:"D", 4: "E"}
  options = []
  for pred in prediction["labels"]:
    label_index = labels.index(pred)
    options.append(index_to_option[label_index])
  options = " ".join(options[:3])
  score += calculate_map(options, answer)

print(f"Average MAP@3 score of RAG pipeline across 20 rows: {score/20:.3f}")


  0%|          | 0/20 [00:00<?, ?it/s]

Average MAP@3 score of RAG pipeline across 20 rows: 0.975
